# Trip Extraction — Day-by-Day DuckDB

Processes one day at a time to keep memory footprint small and constant.
Skips dates that already have parquet output (resumable).

## Trip logic
- Poll interval: ~450 seconds
- **Cell handover** (cell_id changes between consecutive polls) = vehicle moving
- **Stationary** = same cell for >= `STATIONARY_POLLS` consecutive polls
- **Trip** = continuous run of handovers between two stationary periods

```
polls:  [C1] [C2] [C3] [C3] [C3] [C4] [C5] [C5] [C5]
        move move move same same move move same same
        |------ TRIP 1 ------|  park  |--- TRIP 2 ---|
```

In [1]:
## Config

import duckdb
import os
from pathlib import Path

DB_PATH   = "/home/jovyan/data/analytics.duckdb"
STAGE_DIR = Path("/home/jovyan/data/stage/trips")
TMP_DIR   = "/home/jovyan/data/tmp"

# How many consecutive same-cell polls = stationary
STATIONARY_POLLS = 2      # 2 x 450s = 15 min

# Minimum cell handovers for a trip to be kept
MIN_HANDOVERS = 2

# Minimum distinct cells visited — filters cell-edge oscillation
MIN_CELLS = 3

MAX_FILES = 30

STAGE_DIR.mkdir(parents=True, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

print(f"STATIONARY_POLLS={STATIONARY_POLLS}, MIN_HANDOVERS={MIN_HANDOVERS}, MIN_CELLS={MIN_CELLS}")

STATIONARY_POLLS=2, MIN_HANDOVERS=2, MIN_CELLS=3


In [2]:
## Connect and configure DuckDB

con = duckdb.connect(DB_PATH)
con.execute(f"SET memory_limit='8GB'")
con.execute(f"SET temp_directory='{TMP_DIR}'")
con.execute("SET preserve_insertion_order=false")
con.execute("SET threads=4")

## Rebuild handover_events from parquet
#con.execute("DROP TABLE IF EXISTS handover_events")
#con.execute("""
#    CREATE TABLE handover_events AS
#    SELECT *
#    FROM read_parquet('/home/jovyan/data/stage/handover_events/**/*.parquet')
#""")

summary = con.execute("""
    SELECT
        COUNT(*)                   AS total_rows,
        COUNT(DISTINCT vehicle_id) AS vehicles,
        MIN(event_date)            AS first_date,
        MAX(event_date)            AS last_date,
        COUNT(DISTINCT event_date) AS n_dates
    FROM handover_events
""").df()
print(summary.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  vehicles first_date  last_date  n_dates
  913539144     99610 2025-01-01 2025-11-01      304


In [3]:
## Get list of dates to process

dates = [
    row[0] for row in
    con.execute("SELECT DISTINCT event_date FROM handover_events ORDER BY event_date").fetchall()
]

print(f"Dates to process: {len(dates)}")
print(f"  First: {dates[0]}")
print(f"  Last:  {dates[-1]}")

Dates to process: 304
  First: 2025-01-01
  Last:  2025-11-01


In [4]:
## Process one day at a time

import time

total_trips = 0
skipped = 0
run_start = time.time()

for i, event_date in enumerate(dates, 1):

    # --- Resumable: skip if parquet already written for this date ---
    out_dir = STAGE_DIR / f"event_date={event_date}"
    if out_dir.exists() and any(out_dir.glob("*.parquet")):
        skipped += 1
        print(f"[{i}/{len(dates)}] SKIP {event_date} (already done)")
        continue

    day_start = time.time()

    # --- Step A: build trips_raw for this date only ---
    con.execute("DROP TABLE IF EXISTS trips_raw")
    con.execute(f"""
    CREATE TABLE trips_raw AS
    WITH
    ranked AS (
        SELECT
            vehicle_id, imsi, event_ts, cell_id, rat, event_date,
            LAG(cell_id) OVER (PARTITION BY vehicle_id ORDER BY event_ts) AS prev_cell,
            CASE
                WHEN cell_id != LAG(cell_id) OVER (PARTITION BY vehicle_id ORDER BY event_ts)
                  OR LAG(cell_id) OVER (PARTITION BY vehicle_id ORDER BY event_ts) IS NULL
                THEN 1 ELSE 0
            END AS cell_changed
        FROM handover_events
        WHERE event_date = '{event_date}'
    ),
    with_run AS (
        SELECT *,
            SUM(cell_changed) OVER (PARTITION BY vehicle_id ORDER BY event_ts) AS cell_run_id
        FROM ranked
    ),
    with_poll_count AS (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY vehicle_id, cell_run_id ORDER BY event_ts) AS same_cell_poll_count
        FROM with_run
    ),
    with_stationary AS (
        SELECT *,
            same_cell_poll_count >= {STATIONARY_POLLS} AS stationary,
            LAG(same_cell_poll_count >= {STATIONARY_POLLS})
                OVER (PARTITION BY vehicle_id ORDER BY event_ts) AS prev_stationary
        FROM with_poll_count
    ),
    with_trip_flags AS (
        SELECT *,
            CASE
                WHEN cell_changed = 1
                 AND (prev_stationary OR prev_stationary IS NULL)
                THEN 1 ELSE 0
            END AS trip_start_flag
        FROM with_stationary
    ),
    with_trip_seq AS (
        SELECT *,
            SUM(trip_start_flag) OVER (PARTITION BY vehicle_id ORDER BY event_ts) AS trip_seq,
            NOT stationary AS in_trip
        FROM with_trip_flags
    )
    SELECT *,
        CASE WHEN NOT stationary
            THEN vehicle_id || '_trip_' || LPAD(CAST(trip_seq AS VARCHAR), 3, '0')
            ELSE NULL
        END AS trip_id
    FROM with_trip_seq
    """)

    # --- Step B: aggregate trip summaries ---
    con.execute("DROP TABLE IF EXISTS trip_agg")
    con.execute("""
    CREATE TABLE trip_agg AS
    SELECT
        vehicle_id, imsi, trip_id, trip_seq, event_date,
        MIN(event_ts)                                      AS trip_start,
        MAX(event_ts)                                      AS trip_end,
        COUNT(*)                                           AS n_events,
        COUNT(DISTINCT cell_id)                            AS n_cells,
        SUM(CASE WHEN cell_changed = 1 THEN 1 ELSE 0 END) AS n_handovers,
        FIRST(cell_id ORDER BY event_ts)                   AS first_cell,
        LAST(cell_id ORDER BY event_ts)                    AS last_cell,
        MODE(rat)                                          AS dominant_rat
    FROM trips_raw
    WHERE in_trip
    GROUP BY vehicle_id, imsi, trip_id, trip_seq, event_date
    """)

    # --- Step C: free trips_raw, build cell sequences ---
    con.execute("DROP TABLE IF EXISTS trips_raw")

    con.execute("DROP TABLE IF EXISTS cell_sequences")
    con.execute(f"""
    CREATE TABLE cell_sequences AS
    SELECT
        h.vehicle_id || '_trip_' || LPAD(CAST(t.trip_seq AS VARCHAR), 3, '0') AS trip_id,
        STRING_AGG(h.cell_id, ' -> ' ORDER BY h.event_ts) AS cell_sequence
    FROM handover_events h
    JOIN trip_agg t
        ON  h.vehicle_id = t.vehicle_id
        AND h.event_ts  >= t.trip_start
        AND h.event_ts  <= t.trip_end
    WHERE h.event_date = '{event_date}'
    GROUP BY 1
    """)

    # --- Step D: join and apply filters, write parquet ---
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / "trips.parquet"

    n_trips = con.execute(f"""
    COPY (
        SELECT
            a.vehicle_id, a.imsi, a.event_date, a.trip_id, a.trip_seq,
            a.trip_start, a.trip_end,
            ROUND(DATEDIFF('second', a.trip_start, a.trip_end) / 60.0, 1) AS duration_minutes,
            a.n_handovers, a.n_cells, a.n_events,
            a.first_cell, a.last_cell, a.dominant_rat,
            s.cell_sequence
        FROM trip_agg a
        LEFT JOIN cell_sequences s ON a.trip_id = s.trip_id
        WHERE a.trip_id    IS NOT NULL
          AND a.n_handovers >= {MIN_HANDOVERS}
          AND a.n_cells     >= {MIN_CELLS}
        ORDER BY vehicle_id, trip_start
    ) TO '{out_file}' (FORMAT PARQUET)
    """).fetchone()[0]

    # --- Cleanup working tables ---
    con.execute("DROP TABLE IF EXISTS trip_agg")
    con.execute("DROP TABLE IF EXISTS cell_sequences")

    total_trips += n_trips
    elapsed = time.time() - day_start
    print(f"[{i}/{len(dates)}] {event_date} → {n_trips:,} trips ({elapsed:.1f}s)")

total_elapsed = time.time() - run_start
print(f"\nDone. {len(dates) - skipped} days processed, {skipped} skipped.")
print(f"Total trips written: {total_trips:,}")
print(f"Total runtime: {total_elapsed/60:.1f} min")

[1/304] SKIP 2025-01-01 (already done)
[2/304] SKIP 2025-01-02 (already done)
[3/304] SKIP 2025-01-03 (already done)
[4/304] SKIP 2025-01-04 (already done)
[5/304] SKIP 2025-01-05 (already done)
[6/304] SKIP 2025-01-06 (already done)
[7/304] SKIP 2025-01-07 (already done)
[8/304] SKIP 2025-01-08 (already done)
[9/304] SKIP 2025-01-09 (already done)
[10/304] SKIP 2025-01-10 (already done)
[11/304] SKIP 2025-01-11 (already done)
[12/304] SKIP 2025-01-12 (already done)
[13/304] SKIP 2025-01-13 (already done)
[14/304] SKIP 2025-01-14 (already done)
[15/304] SKIP 2025-01-15 (already done)
[16/304] SKIP 2025-01-17 (already done)
[17/304] SKIP 2025-01-18 (already done)
[18/304] SKIP 2025-01-19 (already done)
[19/304] SKIP 2025-01-20 (already done)
[20/304] SKIP 2025-01-21 (already done)
[21/304] SKIP 2025-01-22 (already done)
[22/304] SKIP 2025-01-23 (already done)
[23/304] SKIP 2025-01-24 (already done)
[24/304] SKIP 2025-01-25 (already done)
[25/304] SKIP 2025-01-26 (already done)
[26/304] 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[149/304] 2025-05-30 → 76,143 trips (20.9s)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[150/304] 2025-05-31 → 66,576 trips (22.8s)


...

[303/304] 2025-10-31 → 198,881 trips (63.8s)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[304/304] 2025-11-01 → 155,058 trips (58.3s)

Done. 156 days processed, 148 skipped.
Total trips written: 8,904,766
Total runtime: 35.8 min


In [5]:
## Register trips table in DuckDB from all parquet files

con.execute("DROP TABLE IF EXISTS trips")
con.execute(f"""
    CREATE TABLE trips AS
    SELECT * FROM read_parquet('{STAGE_DIR}/**/*.parquet')
""")

con.execute("""
    SELECT
        COUNT(*)                       AS n_trips,
        COUNT(DISTINCT vehicle_id)     AS vehicles,
        SUM(n_handovers)               AS total_handovers,
        ROUND(AVG(duration_minutes),1) AS avg_duration_min,
        MIN(event_date)                AS first_date,
        MAX(event_date)                AS last_date
    FROM trips
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_trips,vehicles,total_handovers,avg_duration_min,first_date,last_date
0,13024324,96880,70977888.0,38.2,2025-01-01,2025-11-01


In [6]:
## Sanity checks

print("=== Top 10 most-active vehicles ===")
con.execute("""
    SELECT vehicle_id,
           COUNT(*)         AS n_trips,
           SUM(n_handovers) AS total_handovers
    FROM trips
    GROUP BY vehicle_id
    ORDER BY total_handovers DESC
    LIMIT 10
""").df()

=== Top 10 most-active vehicles ===


,vehicle_id,n_trips,total_handovers
0,328657F374AEA61ABE5277C7CED9824E,3820,24481.0
1,79335A7796FD7F70CE3A2A208D9002EC,1568,23181.0
2,C76EE09311EDCCA55A4A9E89F176BD45,3485,21805.0
3,0349865D2EADAC239038E9D9136708D8,3363,21117.0
4,B7F65B2DE6F7BF91D7F9C3D5D2F38868,3412,20940.0
5,EB8CDD68A5F52E62C17816178BC53096,3088,19970.0
6,B151A689C48684D8B7F1BFB2CDB322C9,1512,19571.0
7,225DC783A88D79F028CF21FE5D1F5213,3048,19341.0
8,A878BCE63F7A8E33A9C634E17D80761C,2943,19175.0
9,B7F65B2DE6F7BF9117FE34FC6D1A5B8C,1958,19098.0


In [7]:
## Inspect most active vehicle

top_vehicle = con.execute("""
    SELECT vehicle_id FROM trips
    GROUP BY vehicle_id
    ORDER BY SUM(n_handovers) DESC
    LIMIT 1
""").fetchone()[0]

print(f"Vehicle: {top_vehicle}")
con.execute(f"""
    SELECT trip_id, trip_start, trip_end, duration_minutes,
           n_handovers, n_cells, cell_sequence
    FROM trips
    WHERE vehicle_id = '{top_vehicle}'
    ORDER BY trip_start
    LIMIT 20
""").df()

Vehicle: 328657F374AEA61ABE5277C7CED9824E


,trip_id,trip_start,trip_end,duration_minutes,n_handovers,n_cells,cell_sequence
0,328657F374AEA61ABE5277C7CED9824E_trip_002,2025-03-04 21:15:23.788,2025-03-04 21:47:42.986,32.3,5.0,5,105504441 -> 105504273 -> 105504266 -> 1055044...
1,328657F374AEA61ABE5277C7CED9824E_trip_004,2025-03-04 22:19:59.650,2025-03-04 22:36:08.641,16.2,3.0,3,105504266 -> 105504441 -> 105504435
2,328657F374AEA61ABE5277C7CED9824E_trip_005,2025-03-04 22:52:16.397,2025-03-04 23:08:26.519,16.2,3.0,3,105504280 -> 105504435 -> 105504266
3,328657F374AEA61ABE5277C7CED9824E_trip_006,2025-03-04 23:24:34.196,2025-03-04 23:48:47.015,24.2,4.0,3,105504280 -> 105504435 -> 105504266 -> 105504435
4,328657F374AEA61ABE5277C7CED9824E_trip_007,2025-03-05 02:38:16.582,2025-03-05 03:02:29.409,24.2,4.0,3,105504266 -> 105504435 -> 105504441 -> 105504266
5,328657F374AEA61ABE5277C7CED9824E_trip_009,2025-03-05 04:23:14.190,2025-03-05 04:55:30.826,32.3,5.0,5,105504441 -> 105504280 -> 105504435 -> 1055042...
6,328657F374AEA61ABE5277C7CED9824E_trip_011,2025-03-05 05:52:01.531,2025-03-05 06:40:25.831,48.4,7.0,5,105504435 -> 105504266 -> 105504273 -> 1055042...
7,328657F374AEA61ABE5277C7CED9824E_trip_012,2025-03-05 06:56:34.788,2025-03-05 07:44:59.262,48.4,7.0,5,105504266 -> 105504273 -> 105504280 -> 1055042...
8,328657F374AEA61ABE5277C7CED9824E_trip_013,2025-03-05 08:01:09.543,2025-03-05 08:17:19.587,16.2,3.0,3,105504280 -> 105504435 -> 105504273
9,328657F374AEA61ABE5277C7CED9824E_trip_015,2025-03-05 09:05:42.654,2025-03-05 09:21:50.351,16.1,3.0,3,105504266 -> 105504280 -> 105504435


In [8]:
con.execute(f"""
    SELECT trip_id, trip_start, trip_end, duration_minutes,
           n_handovers, n_cells, cell_sequence
    FROM trips
    WHERE vehicle_id = 'A2FCC9AF8E480E683F655628A3E5E38D'
    ORDER BY trip_start
    LIMIT 20
""").df()

,trip_id,trip_start,trip_end,duration_minutes,n_handovers,n_cells,cell_sequence
0,A2FCC9AF8E480E683F655628A3E5E38D_trip_007,2025-05-15 21:29:19.993,2025-05-15 22:01:31.265,32.2,5.0,5,182035976 -> 182043146 -> 182074122 -> 1820485...
1,A2FCC9AF8E480E683F655628A3E5E38D_trip_008,2025-05-15 22:17:36.970,2025-05-15 22:33:42.468,16.1,3.0,3,182074136 -> 182311177 -> 182043158
2,A2FCC9AF8E480E683F655628A3E5E38D_trip_002,2025-05-16 00:42:40.725,2025-05-16 02:09:57.997,87.3,12.0,9,182043144 -> 182043158 -> 182035976 -> 1820175...
3,A2FCC9AF8E480E683F655628A3E5E38D_trip_039,2025-05-17 20:44:33.790,2025-05-17 21:40:55.089,56.4,8.0,4,183462719 -> 181786391 -> 182309910 -> 1823098...
4,A2FCC9AF8E480E683F655628A3E5E38D_trip_042,2025-05-17 22:53:29.677,2025-05-17 23:17:37.679,24.1,4.0,4,182309896 -> 181786390 -> 182043144 -> 182035991
5,A2FCC9AF8E480E683F655628A3E5E38D_trip_027,2025-05-19 11:26:01.247,2025-05-19 12:14:17.806,48.3,7.0,5,182035990 -> 182043158 -> 182311190 -> 1820551...
6,A2FCC9AF8E480E683F655628A3E5E38D_trip_030,2025-05-19 13:42:54.768,2025-05-19 13:59:00.379,16.1,3.0,3,182058506 -> 182074121 -> 182074135
7,A2FCC9AF8E480E683F655628A3E5E38D_trip_039,2025-05-19 18:32:40.767,2025-05-19 19:12:54.334,40.2,6.0,4,182074121 -> 182074129 -> 182074135 -> 1820741...
8,A2FCC9AF8E480E683F655628A3E5E38D_trip_040,2025-05-19 19:29:08.727,2025-05-19 19:45:23.166,16.3,3.0,3,182311690 -> 182061078 -> 182061064
9,A2FCC9AF8E480E683F655628A3E5E38D_trip_041,2025-05-19 20:01:28.827,2025-05-19 20:49:45.407,48.3,7.0,6,182061080 -> 182061066 -> 182061065 -> 1820431...


In [9]:
# Export trips sample to Excel
result = con.execute("""
    SELECT trip_id, trip_start, trip_end, duration_minutes,
           n_handovers, n_cells, cell_sequence
    FROM trips
    WHERE vehicle_id = 'A2FCC9AF8E480E683F655628A3E5E38D'
    ORDER BY trip_start
""").df()

result.to_excel('/home/jovyan/data/stage/trips/A2FCC9AF8E480E683F655628A3E5E38D_all_trips.xlsx', index=False)
print(f"Exported {len(result)} rows")

Exported 532 rows


In [10]:
con.execute("""
    SELECT MIN(event_date), MAX(event_date), COUNT(DISTINCT event_date)
    FROM trips
""").df()

,min(event_date),max(event_date),count(DISTINCT event_date)
0,2025-01-01,2025-11-01,299
